# Train

> GPUs pulling all the weight

In [ ]:
#| default_exp train

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, torch.nn as nn, lightning.pytorch as pl, warnings, math

from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingWarmRestarts

### PatchTFT Single Outcome Prediction

In [ ]:
#| export
class PatchTFTSingleOutcomeLightning(pl.LightningModule): #encoder for linear probing head that predict EDS
    def __init__(self, #constructor that contains or calls the main dataset setup code
                linear_probing_head, # model head to linear probe/train
                learning_rate, # desired learning rate, initial learning rate in if one_cycle_scheduler
                train_size, # the training data size (for one_cycle_scheduler=True)
                batch_size, # the batch size (for one_cycle_scheduler=True)
                n_gpus, # number of GPUs
                preloaded_model, # loaded pretrained model to use for linear probing
                metrics={}, # name:function for metrics to log
                fine_tune=False, # indicator to fine tune encoder or freeze encoder weights and perform linear probing
                class_weights=None, # weights of classes to use in CE loss fxn
                epochs=100, # number of epochs for one_cycle_scheduler
                scheduler_type='OneCycle',
                optimizer_type='AdamW',
                weight_decay=0., # weight decay for Adam optimizer
                use_weight_decay_scheduler=False, # use a weight decay scheduler
                final_weight_decay=0.01, # final weight decay for the weight decay scheduler
                scheduler_kwargs={}, # kwargs for the scheduler
                transforms=None, # transforms to apply to the data
                mixup_callback=None, # mixup callback to apply to the data
                ):
        super().__init__()
        self.encoder = preloaded_model
        self.scheduler_type = scheduler_type
        if self.scheduler_type is not None:
            assert self.scheduler_type.lower() in ['onecycle', 'cosineannealingwarmrestarts'], "scheduler must be either OneCycle, CosineAnnealingWarmRestarts, or None"
        self.weight_decay = weight_decay
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.train_size = train_size
        self.batch_size = batch_size * n_gpus
        self.ipe = self.train_size//self.batch_size
        self.metrics = nn.ModuleDict(metrics)
        self.ipe = self.train_size//self.batch_size
        self.class_weights = class_weights
        self.fine_tune = fine_tune
        self.use_weight_decay_scheduler = use_weight_decay_scheduler
        self.scheduler_kwargs = scheduler_kwargs
        self.final_weight_decay = final_weight_decay
        self.optimizer_type = optimizer_type
        self.transforms = transforms
        self.mixup_callback = mixup_callback
        if not self.fine_tune:
            self.encoder.freeze() # freeze the encoder weights (sets to eval mode)
            if hasattr(self.encoder, 'pretrain'):
                setattr(self.encoder, 'pretrain', False)
                setattr(self.encoder.model, 'pretrain', False)
        # Adjust the output layer for binary classification
        self.feedforward = linear_probing_head # model head to linear probe/train
        self.save_hyperparameters(ignore=['linear_probing_head', 'preloaded_model'])

    def forward(self, x):
        x = self.encoder(x) # [bs, n_channels, d_model, n_ffts/n_patches]
        if isinstance(x, tuple):
            # contrastive model
            x = x[0]
        if torch.isnan(x).any():
            warnings.warn("NaN values in input to feedforward layer")
        x = self.feedforward(x) # [bs, n_classes, pred_len_seconds]
        return x # Ensure the output is of shape [batch_size]
    
    def on_train_batch_start(self, batch, batch_idx):
        # update weight decay
        if self.use_weight_decay_scheduler:
            step = self.global_step
            T_max = int(self.ipe * self.epochs)
            progress = step / T_max
            new_wd = self.final_weight_decay + (self.weight_decay - self.final_weight_decay) * 0.5 * (1. + math.cos(math.pi * progress))

            if self.final_weight_decay <= self.weight_decay:
                new_wd = max(self.final_weight_decay, new_wd)
            else:
                new_wd = min(self.final_weight_decay, new_wd)

            for group in self.optimizer.param_groups:
                if ('WD_exclude' not in group) or not group['WD_exclude']:
                    group['weight_decay'] = new_wd

    def predict_step(self, batch, batch_idx, dataloader_idx=0): #does the order matter? should it be after test_step()
        x,y = batch
        preds = self(x)
        return preds, y

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        if self.transforms is not None:
            batch = self.transforms(batch)
        if self.mixup_callback is not None:
            batch = self.mixup_callback(batch)
        x, y = batch
        x = self(x)  # Ensure output is [batch_size]
        loss = nn.BCEWithLogitsLoss(pos_weight=self.class_weights.to(x.device) if self.class_weights is not None else None)
        loss_val = loss(x,y.float())
        self.log('train_loss', loss_val, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        return loss_val

    def validation_step(self, batch, batch_idx):
        x, y = batch
        x = self(x)  # Ensure output is [batch_size]
        loss = nn.BCEWithLogitsLoss(pos_weight=self.class_weights.to(x.device) if self.class_weights is not None else None)
        loss_val = loss(x,y.float())
        self.log('val_loss', loss_val, prog_bar=True, on_step=True, on_epoch=True, sync_dist=True)
        
        x_probs = torch.sigmoid(x)
        for metric in self.metrics:
            self.metrics[metric].update(x_probs, y.long())
    
    def on_validation_epoch_end(self):
        for name, metric in self.metrics.items():
            metric_val = metric.compute()
            self.log(f'val_{name}', metric_val, prog_bar=True, on_step=False, on_epoch=True, sync_dist=True)
            metric.reset()
    
    def on_validation_epoch_start(self): #start of a validation epoch during the training or evaluation of a model in a deep learning framework
        #do you need to end it?
        torch.cuda.empty_cache() #releases all unused memory cached by the CUDA driver 
        #for the PyTorch process to potentially reduce the memory footprint on the GPU

    def configure_optimizers(self):
        param_groups = [ # exclude bias and layer norm parameters from weight decay
           {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' not in n) and (len(p.shape) != 1))
            }, {
                'params': (p for n, p in self.feedforward.named_parameters()
                        if ('bias' in n) or (len(p.shape) == 1)),
                'WD_exclude': True,
                'weight_decay': 0,
            },
        ]
        self.optimizer = torch.optim.AdamW(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0) if self.optimizer_type.lower() == 'adamw' else\
                     torch.optim.Adam(param_groups, lr=self.learning_rate, weight_decay=self.weight_decay if not self.use_weight_decay_scheduler else 0)
        if self.scheduler_type.lower() == 'onecycle':
            scheduler = OneCycleLR(self.optimizer, epochs=self.epochs, steps_per_epoch=self.ipe, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'step'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        elif self.scheduler_type.lower() == 'cosineannealingwarmrestarts':
            scheduler = CosineAnnealingWarmRestarts(self.optimizer, **self.scheduler_kwargs)
            lr_scheduler = {'scheduler': scheduler, 'interval': 'epoch'}
            return {'optimizer': self.optimizer, 'lr_scheduler': lr_scheduler}
        else:
            return self.optimizer

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()